Why SAC?

We already know PPO, so let's use it as the reference point.

1. The problem with PPO

PPO is on-policy.

Suppose PPO collects:

Actor π_old
    ↓
1000 transitions
    ↓
update
    ↓
new actor π_new

Those old 1000 transitions were generated by the old policy.

Once we move to π_new, we generally throw that experience away.

So if collecting experience is expensive:

PPO is wasting a lot of potentially useful data.

2. SAC's key idea

SAC says:

"Why throw away the experience? Store it and reuse it."

So:

Environment
    ↓
experience
    ↓
Replay Buffer
    ↓
┌───────────────┐
│ old + new data│
└───────────────┘
    ↓
sample random batches
    ↓
update SAC

The same transition can potentially be used many times.

That's the first major difference:

	PPO	SAC
Learning	On-policy	Off-policy
Experience	Mostly fresh	Replayable
Replay buffer	❌	✅
Sample efficiency	Lower	Higher
3. But why not just make an off-policy actor-critic?

Because SAC has another major idea:

Entropy

Normally RL says:

Choose actions that maximize expected reward.

SAC says:

Choose actions that maximize reward while also maintaining useful randomness/exploration.

Conceptually:

SAC objective
=
expected reward
+
α × entropy

So the policy isn't encouraged to become deterministic too quickly.

4. Why is that useful?

Imagine a robot has discovered:

Action A → reward 8
Action B → reward 7
Action C → reward 2

A normal policy might quickly become:

A: 99%
B: 1%
C: 0%

SAC says:

"Don't collapse your exploration too quickly. There may be something better you haven't discovered."

So it might maintain:

A: 70%
B: 25%
C: 5%

while learning.

Eventually, as it becomes confident, the exploration pressure can decrease.

5. So why SAC?

The two big motivations are:

              SAC
             /   \
            /     \
   Replay data    Entropy
   reuse          exploration
      ↓               ↓
sample efficient   robust exploration

And this makes SAC particularly useful for continuous-control problems, where exploring a large continuous action space can be difficult.

The mental model to remember

You've now seen three major approaches:

DQN
→ learn which action is valuable

PPO
→ directly improve the policy using fresh experience

SAC
→ improve the policy using reusable experience
  while explicitly encouraging exploration

Entropy & Temperature

The key thing to understand first:

SAC doesn't only ask “how good is this action?” It also asks “how uncertain/exploratory should my policy remain?”

Suppose at a state:

Action A → Q = 10
Action B → Q = 9

A pure reward-maximizing policy wants:

A ≈ 100%
B ≈ 0%

SAC adds entropy, which rewards having a spread-out probability distribution.

So SAC balances:

high Q          → exploit
high entropy    → explore

The objective is conceptually:

maximize  Q(s,a) + α H(π(.|s))
What is α?

α is the temperature.

It controls how much SAC cares about exploration:

α HIGH
→ entropy matters a lot
→ more exploration

α LOW
→ reward/Q matters more
→ more exploitation

So:

             α
             ↓
      ┌──────────────┐
      │ exploration  │
      │   pressure   │
      └──────────────┘
Why make α learnable?

Because we don't want to manually decide:

"Use exactly 0.2 exploration forever."

Early training may need lots of exploration:

α ↑

Later, once the policy understands the environment:

α ↓

SAC can automatically adjust it toward a target entropy.

That's the important idea.

One subtle point

Entropy isn't literally "randomness = good."

It's:

SAC rewards maintaining uncertainty in the policy when that helps exploration.

If two actions are both good, SAC has a reason to keep probability on both instead of prematurely collapsing onto one.

Remember this
Q value
→ "Is this action good?"

Entropy
→ "Is my policy exploring enough?"

α
→ "How much should I care about that exploration?"